# Phase 4B — Adversarial Training
**Goal:** fine-tune the pretrained generator using pixel, perceptual, and adversarial losses.

The generator starts from the best full-DIV2K checkpoint. The discriminator starts with random weights and both Adam optimizers start fresh. Each run saves its own generator, discriminator, optimizer states, and history on Drive.

**Pilot:** 3 epochs, 800 training images, 50 validation crops, batch size 8, and float32. That gives 300 alternating G/D updates. The held-out test set is reserved for phase 5.

Use a T4 runtime and keep Drive mounted. The first run downloads about 548 MB of pretrained VGG19 weights. Training time depends on the runtime and Drive, so use the measured first-epoch duration to estimate the remaining two epochs.

The loss is inspired by [Ledig et al. (CVPR 2017)](https://openaccess.thecvf.com/content_cvpr_2017/html/Ledig_Photo-Realistic_Single_Image_CVPR_2017_paper.html). The pixel term and weights are choices for this pilot.


## 1. Update the project
The cell fast-forwards an existing clean main checkout. Local tracked edits are preserved: if present, the update stops instead of discarding them.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path("/content/SRGAN-Image-Super-Resolution")
REPO_URL = "https://github.com/ennaguiyoussef/SRGAN-Image-Super-Resolution.git"
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    if not (PROJECT_ROOT / ".git").is_dir():
        raise RuntimeError("Project folder exists but is not a Git checkout.")
    changes = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "status", "--porcelain", "--untracked-files=no"],
        text=True,
    )
    if changes.strip():
        raise RuntimeError("Save local tracked edits before updating this checkout.")
    branch = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "branch", "--show-current"], text=True
    ).strip()
    if branch != "main":
        raise RuntimeError("Select the main branch before running this notebook.")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
importlib.invalidate_caches()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Project checkout ready.")


## 2. Read settings and verify the GPU
The generator's objective combines:

- **Pixel MSE × 1.0** for similarity to the HR pixels.
- **VGG feature MSE × 0.01** for similarity of the image features.
- **Adversarial BCE × 0.001** for a signal from the discriminator.

The discriminator uses binary cross-entropy with logits for real HR and generated SR images. Separate Adam optimizers use 5e-5 for G and 1e-4 for D.

The perceptual term uses frozen VGG19 features after `relu2_2`. RGB values are mapped from [-1, 1] to [0, 1] and normalized with the pretrained weights' ImageNet mean and standard deviation. The spatial size stays 96×96 for both inputs.


In [ ]:
import yaml
import torch
from src.setup import environment_report, load_config, seed_everything

config = load_config(PROJECT_ROOT / "configs/srgan.yaml")
gan_config = yaml.safe_load((PROJECT_ROOT / "configs/adversarial.yaml").read_text())
report = environment_report()
print(json.dumps(report, indent=2))
if not report["cuda_available"]:
    raise RuntimeError("Select a GPU runtime before training.")
device = torch.device("cuda:0")
seed_everything(config["seed"])
print(json.dumps(gan_config, indent=2))


## 3. Mount Drive and restore training and validation data
Reuse the phase-2 archives and the same manifest. The training set contains all 800 images. Validation keeps the same 50 center crops used for generator pretraining.


In [ ]:
from google.colab import drive
from src.setup import prepare_directories
from src.data import extract_hr_archive, split_manifest
from src.dataset import SuperResolutionDataset

drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/SRGAN-Image-Super-Resolution")
paths = prepare_directories(PROJECT_ROOT, OUTPUT_ROOT)
DATA_ROOT = paths["data"] / "DIV2K"
manifest_path = OUTPUT_ROOT / "splits" / "div2k_seed42.json"
if not manifest_path.is_file():
    raise RuntimeError("Complete phase 2 first: split manifest is missing.")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if manifest != split_manifest(config["seed"]):
    raise RuntimeError("Split manifest does not match the current seed.")
train_names = manifest["train"]
assert len(train_names) == 800
validation_names = manifest["validation"]
for folder, names in [("DIV2K_train_HR", train_names), ("DIV2K_valid_HR", validation_names)]:
    missing = [name for name in names if not (DATA_ROOT / name).is_file()]
    if missing:
        archive = OUTPUT_ROOT / "dataset_cache" / f"{folder}.zip"
        if not archive.is_file():
            raise RuntimeError(f"Missing archive {archive.name}. Complete phase 2 first.")
        extract_hr_archive(archive, DATA_ROOT, folder, [int(Path(name).stem) for name in missing])

train_dataset = SuperResolutionDataset(
    DATA_ROOT, train_names, hr_size=config["hr_crop_size"], scale=config["scale_factor"], training=True,
)
validation_dataset = SuperResolutionDataset(
    DATA_ROOT, validation_names, hr_size=config["hr_crop_size"], scale=config["scale_factor"], training=False,
)
print("Training images:", len(train_dataset))
print("Validation images:", len(validation_dataset))
print("Held-out test images are not loaded.")


## 4. Initialize the models
A new run loads the best pretrained generator weights from the full-data `last.pt` checkpoint. On rerun the GAN checkpoint takes priority and restores both models and both Adam optimizers.

VGG19 runs in evaluation mode with frozen weights. Gradients still flow from its features back to the generated image. The pretrained weights cache is stored on Drive for later sessions.


In [ ]:
import hashlib
import re
from src.models import Generator, Discriminator
from src.adversarial import VGGFeatureLoss

for key in ("run_name", "source_run"):
    if not re.fullmatch(r"[A-Za-z0-9_-]+", gan_config[key]):
        raise ValueError(f"{key} must contain only letters, numbers, underscores, or hyphens.")
RUN_ID = gan_config["run_name"]
RUN_DIR = paths["checkpoints"] / RUN_ID
RESULTS_DIR = paths["results"] / RUN_ID
SOURCE_CHECKPOINT = paths["checkpoints"] / gan_config["source_run"] / "last.pt"
if not (RUN_DIR / "last.pt").is_file() and not SOURCE_CHECKPOINT.is_file():
    raise FileNotFoundError("Complete the full-data phase 4A run and restore its last.pt on Drive.")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

seed_everything(config["seed"])
generator = Generator()
discriminator = Discriminator()
torch.hub.set_dir(str(OUTPUT_ROOT / "torch_cache"))
feature_loss = VGGFeatureLoss()
experiment = {
    "phase": "4B",
    "scale_factor": config["scale_factor"], "hr_crop_size": config["hr_crop_size"],
    "manifest_sha256": hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
    "train_names": train_names, "validation_names": validation_names,
    "degradation": "Pillow bicubic after HR crop", "lr_range": [0, 1], "hr_range": [-1, 1],
    "feature_network": "torchvision VGG19 IMAGENET1K_V1 relu2_2",
}
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Source checkpoint:", SOURCE_CHECKPOINT)
print("GAN run directory:", RUN_DIR)
print("GAN resume checkpoint exists:", (RUN_DIR / "last.pt").is_file())


## 5. Alternate discriminator and generator updates
1. **Update D.** Compare HR and detached SR. This updates D without passing gradients to G.
2. **Update G.** Freeze D parameters and statistics. Combine pixel, VGG, and adversarial errors and backpropagate to G.

After each complete epoch, `last.pt` stores both models, their Adam states, and the metrics. A partially completed epoch is repeated after interruption. To continue, raise the total epoch target and keep the other settings.

The baseline at epoch 0 uses the pretrained generator. The exported GAN candidate comes from the trained epoch with the lowest validation MSE. Compare its visual detail and artifacts alongside the scores.


In [ ]:
from src.adversarial import fit_adversarial

state = fit_adversarial(
    generator, discriminator, feature_loss, train_dataset, validation_dataset,
    RUN_DIR, SOURCE_CHECKPOINT,
    epochs=gan_config["epochs"], batch_size=gan_config["batch_size"],
    generator_lr=gan_config["generator_lr"], discriminator_lr=gan_config["discriminator_lr"],
    loss_weights=gan_config["loss_weights"], gradient_clip_norm=gan_config["gradient_clip_norm"],
    seed=config["seed"], device=device, resume=gan_config["resume"],
    experiment=experiment, source_commit=commit,
)
print("Source best epoch:", state["initialization"]["source_best_epoch"])
print("Completed GAN epoch:", state["epoch"])
print("Best trained GAN epoch by validation MSE:", state["best_epoch"])


## 6. Inspect training and validation curves
The generator loss components use their configured weights in the plot. Validation reports pixel MSE and mean RGB PSNR on the same 50 center crops as phase 4A. The first validation point shows the pretrained generator.

Adversarial training changes the objective. Compare the images as well as PSNR to see how it affects textures.


In [ ]:
import matplotlib.pyplot as plt

history = state["history"]
trained = history[1:]
epochs = [row["epoch"] for row in trained]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for key, label in [("pixel", "Pixel"), ("feature", "VGG feature"), ("adversarial", "Adversarial")]:
    metric = {"pixel": "g_pixel_mse", "feature": "g_feature_mse", "adversarial": "g_adversarial_bce"}[key]
    axes[0, 0].plot(
        epochs, [gan_config["loss_weights"][key] * row[metric] for row in trained],
        marker="o", label=label,
    )
axes[0, 0].set(xlabel="GAN epoch", title="Weighted generator loss components")
axes[0, 0].legend()
axes[0, 1].plot(epochs, [row["d_bce"] for row in trained], marker="o")
axes[0, 1].set(xlabel="GAN epoch", title="Discriminator BCE")
for axis, metric, title in [
    (axes[1, 0], "val_mse", "Validation MSE"),
    (axes[1, 1], "val_psnr_rgb", "Validation RGB PSNR (dB)"),
]:
    axis.plot([row["epoch"] for row in history], [row[metric] for row in history], marker="o")
    axis.set(xlabel="GAN epoch (0 = pretrained)", title=title)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "gan_learning_curves.png", dpi=160)
plt.show()


## 7. Compare reconstruction and textures
The comparison uses LR, bicubic interpolation, the pretrained generator, the selected GAN generator, and HR. Inspect edges, fine textures, and color consistency.

Validation uses RGB 96×96 center crops and mean per-image PSNR without border shaving. These values are comparable with phase 4A on our validation split.


In [ ]:
import numpy as np
from PIL import Image
from src.pretrain import validate
from torch.utils.data import DataLoader

preview_pairs = [validation_dataset[i] for i in range(3)]
preview_lr = torch.stack([pair[0] for pair in preview_pairs]).to(device)
generator.load_state_dict(state["initial_generator"])
generator.eval()
with torch.inference_mode():
    initial_sr = generator(preview_lr).cpu()
generator.load_state_dict(state["best_model"])
generator.eval()
with torch.inference_mode():
    best_sr = generator(preview_lr).cpu()

def rgb_image(tensor):
    return ((tensor + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(3, 5, figsize=(16, 10))
for row, (lr, hr) in enumerate(preview_pairs):
    lr_image = lr.permute(1, 2, 0).numpy()
    bicubic = Image.fromarray(np.rint(lr_image * 255).astype(np.uint8)).resize(
        (96, 96), Image.Resampling.BICUBIC
    )
    panels = [(lr_image, "LR 24 x 24"), (bicubic, "Bicubic")]
    panels.extend([
        (rgb_image(initial_sr[row]), "Pretrained G"),
        (rgb_image(best_sr[row]), f"GAN G: epoch {state['best_epoch']}"),
        (rgb_image(hr), "HR reference"),
    ])
    for col, (image, title) in enumerate(panels):
        axes[row, col].imshow(image, interpolation="nearest")
        axes[row, col].set_title(title)
        axes[row, col].axis("off")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "gan_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

# Same 50 validation crops and Pillow bicubic policy as the visual baseline.
bicubic_mses, bicubic_psnrs = [], []
for lr, hr in validation_dataset:
    array = np.rint(lr.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    baseline = Image.fromarray(array).resize((96, 96), Image.Resampling.BICUBIC)
    baseline_tensor = torch.from_numpy(np.array(baseline, dtype=np.float32).transpose(2, 0, 1).copy()) / 255
    target = (hr + 1) / 2
    mse01 = (baseline_tensor - target).square().mean().item()
    bicubic_mses.append(mse01 * 4)
    bicubic_psnrs.append(-10 * np.log10(max(mse01, 1e-12)))
best_metrics = validate(
    generator, DataLoader(validation_dataset, batch_size=gan_config["batch_size"], num_workers=0), device
)
print(f"Bicubic: MSE={np.mean(bicubic_mses):.6f}, RGB PSNR={np.mean(bicubic_psnrs):.2f} dB")
print(f"Pretrained baseline: MSE={state['history'][0]['val_mse']:.6f}, RGB PSNR={state['history'][0]['val_psnr_rgb']:.2f} dB")
print(f"Selected GAN G: MSE={best_metrics['val_mse']:.6f}, RGB PSNR={best_metrics['val_psnr_rgb']:.2f} dB")


## 8. Save the report
The JSON report records the pretrained baseline, selected GAN metrics, bicubic metrics, and source checkpoint provenance. Keep the best GAN weights and the full-data pretrained weights for phase 5.


In [ ]:
from src.pretrain import atomic_json_save

report.update({
    "git_commit": commit, "run_id": RUN_ID, "gan_settings": gan_config,
    "completed_epoch": state["epoch"], "best_epoch": state["best_epoch"],
    "pretrained_validation_baseline": state["history"][0],
    "initialization": state["initialization"],
    "best_gan_validation": best_metrics,
    "bicubic_validation": {
        "val_mse": float(np.mean(bicubic_mses)),
        "val_psnr_rgb": float(np.mean(bicubic_psnrs)),
    },
    "validation_protocol": "50 fixed RGB 96x96 center crops; no border shave; mean per-image PSNR",
    "checkpoint": str(RUN_DIR / "last.pt"),
    "best_weights": str(RUN_DIR / "best_gan_generator.pt"),
    "test_used": False,
})
atomic_json_save(report, RESULTS_DIR / "gan_report.json")
print("Phase 4B pilot complete. Validation and checkpoints saved.")
print("Results:", RESULTS_DIR)
print("Next: review GAN metrics and images before final evaluation.")
